!required packages

```bash
%pip3 install rpy2
```

In [7]:
import pandas as pd
import rpy2.robjects as robjects
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter
import pathlib

# set path of the results
path = pathlib.Path("../integrative-drugrep-tb/results/")

In [8]:
# Define the R readRDS function
read_rds = robjects.r['readRDS']

def load_rds_to_df(file_path):
    """Loads an .rds file and converts it to a pandas DataFrame."""
    r_obj = read_rds(str(file_path)) # Ensure path is a string
    with localconverter(robjects.default_converter + pandas2ri.converter):
        return robjects.conversion.get_conversion().rpy2py(r_obj)

# load all dfs

In [9]:
rnaseq_path = path / "RNAseq/04_rank_aggregation"

CMAP_indivSig_TB_RNAseq         = load_rds_to_df(rnaseq_path / "CMAP_indivSig_TB_RNAseq.rds")
Cor_pearson_indivSig_TB_RNAseq  = load_rds_to_df(rnaseq_path / "Cor_pearson_indivSig_TB_RNAseq.rds")
Cor_spearman_indivSig_TB_RNAseq = load_rds_to_df(rnaseq_path / "Cor_spearman_indivSig_TB_RNAseq.rds")
NCS_indivSig_TB_RNAseq          = load_rds_to_df(rnaseq_path / "NCS_indivSig_TB_RNAseq.rds")
Tau_indivSig_TB_RNAseq          = load_rds_to_df(rnaseq_path / "Tau_indivSig_TB_RNAseq.rds")
WCS_indivSig_TB_RNAseq          = load_rds_to_df(rnaseq_path / "WCS_indivSig_TB_RNAseq.rds")


In [10]:
micro_path = path / "microarray/04_rank_aggregation"

CMAP_indivSig_TB_microarray         = load_rds_to_df(micro_path / "CMAP_indivSig_TB_microarray.rds")
Cor_pearson_indivSig_TB_microarray  = load_rds_to_df(micro_path / "Cor_pearson_indivSig_TB_microarray.rds")
Cor_spearman_indivSig_TB_microarray = load_rds_to_df(micro_path / "Cor_spearman_indivSig_TB_microarray.rds")
NCS_indivSig_TB_microarray          = load_rds_to_df(micro_path / "NCS_indivSig_TB_microarray.rds")
Tau_indivSig_TB_microarray          = load_rds_to_df(micro_path / "Tau_indivSig_TB_microarray.rds")
WCS_indivSig_TB_microarray          = load_rds_to_df(micro_path / "WCS_indivSig_TB_microarray.rds")

# example of the df structure

similar to the second screenshot that you sent me

In [25]:
CMAP_indivSig_TB_RNAseq[0]

'vemurafe...,'temsirol...,'fostamat...,...,'vinorelb...,'zafirluk...,'zidovudi...


In [23]:
CMAP_indivSig_TB_RNAseq[1].head()

,unique_pert,cell,min_score
1,vemurafenib,MCF7,-0.653286
2,temsirolimus,MCF7,-0.667176
3,fostamatinib,SKBR3,-0.621939
4,perphenazine,HA1E,-0.633731
5,vorinostat,PHH,-0.358134


In [ ]:
CMAP_indivSig_TB_RNAseq[2].head() # df for the query

,unique_pert,median_min_score
1,vemurafenib,-0.783576
2,temsirolimus,-0.757920
3,fostamatinib,-0.740455
4,perphenazine,-0.721831
5,vorinostat,-0.704421


In [42]:
def extract_drug_list(scores_dict):
	"""
	Extracts the top 20 drugs from a list of dataframes and returns them as a list.

	logic is as follows:
		for each dataframe in the list:
		1. ensure that the `median_min_score` is in ascending order (lowest to highest)
		2. add to results list 
		3. result list has [metric, drug_name, median_min_score] where metric is the name of the dataframe (e.g. CMAP, Cor_pearson, etc.)

	returns df with columns: metric, drug_name, median_min_score
	"""

	results = []

	for metric_name, df in scores_dict.items():
		df_sorted = df.sort_values(by='median_min_score', ascending=True)
		top_20_drugs = df_sorted.head(20)
		for _, row in top_20_drugs.iterrows():
			results.append({
				'metric': metric_name,
				'drug_name': row['unique_pert'],
				'median_min_score': row['median_min_score']
			})
	return pd.DataFrame(results)

In [43]:
# Map the names you want to the specific dataframes
scores_map = {
    "CMAP_indivSig_TB_RNAseq": CMAP_indivSig_TB_RNAseq[2],
    "Cor_pearson_indivSig_TB_RNAseq": Cor_pearson_indivSig_TB_RNAseq[2],
    "Cor_spearman_indivSig_TB_RNAseq": Cor_spearman_indivSig_TB_RNAseq[2],
    "NCS_indivSig_TB_RNAseq": NCS_indivSig_TB_RNAseq[2],
    "Tau_indivSig_TB_RNAseq": Tau_indivSig_TB_RNAseq[2],
    "WCS_indivSig_TB_RNAseq": WCS_indivSig_TB_RNAseq[2],
    "CMAP_indivSig_TB_microarray": CMAP_indivSig_TB_microarray[2],
    "Cor_pearson_indivSig_TB_microarray": Cor_pearson_indivSig_TB_microarray[2],
    "Cor_spearman_indivSig_TB_microarray": Cor_spearman_indivSig_TB_microarray[2],
    "NCS_indivSig_TB_microarray": NCS_indivSig_TB_microarray[2],
    "Tau_indivSig_TB_microarray": Tau_indivSig_TB_microarray[2],
    "WCS_indivSig_TB_microarray": WCS_indivSig_TB_microarray[2]
}

# Run the function
drug_list_df = extract_drug_list(scores_map)


In [44]:
drug_list_df

,metric,drug_name,median_min_score
0,CMAP_indivSig_TB_RNAseq,vemurafenib,-0.783576
1,CMAP_indivSig_TB_RNAseq,temsirolimus,-0.757920
2,CMAP_indivSig_TB_RNAseq,fostamatinib,-0.740455
3,CMAP_indivSig_TB_RNAseq,perphenazine,-0.721831
4,CMAP_indivSig_TB_RNAseq,vorinostat,-0.704421
...,...,...,...
235,WCS_indivSig_TB_microarray,cabergoline,-0.398786
236,WCS_indivSig_TB_microarray,vorinostat,-0.398520
237,WCS_indivSig_TB_microarray,danazol,-0.397047
238,WCS_indivSig_TB_microarray,cimetidine,-0.394135


# add 